In [2]:
import os
import torch
from timm import create_model
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import json

c:\Users\HP\studia\trzeci rok\semestr6\Warsztaty Badawcze\proekt\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In this task, we focus on evaluating the representational quality of different layers in a pretrained Vision Transformer model, specifically the DINO self-supervised model. We extract features from each transformer block and train lightweight linear classifiers (linear probes). This procedure allows us to measure how linearly separable the learned features are at each depth of the network. The primary goal is to understand which layers encode the most useful semantic information for classification tasks. This analysis serves as a foundation for subsequent investigations, such as layer-skipping strategies or representational redundancy detection, aiming to optimize inference without significantly sacrificing model performance.

In [ ]:
import os
import random
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, ConcatDataset
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

base_train_dir = "C:\\Users\\HP\\Downloads\\archive (15)"
train_dirs = [os.path.join(base_train_dir, f"train.X{i}") for i in range(1, 5)]
val_path = os.path.join(base_train_dir, "val.X")

train_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_datasets = [datasets.ImageFolder(train_dir, transform=train_transform) for train_dir in train_dirs]

selected_classes = []
for ds in train_datasets:
    all_classes = ds.classes
    random_classes = random.sample(all_classes, 2)
    selected_classes.extend(random_classes)

print("Selected classes:", selected_classes)

all_class_to_idx = {}
for ds in train_datasets:
    all_class_to_idx.update(ds.class_to_idx)
class_indices = [all_class_to_idx[cls] for cls in selected_classes]

def get_class_subset(dataset, class_indices):
    selected = [i for i, (_, label) in enumerate(dataset.samples) if label in class_indices]
    return Subset(dataset, selected)

train_subsets = [get_class_subset(ds, [ds.class_to_idx[cls] for cls in ds.classes if cls in selected_classes]) for ds in train_datasets]
train_subset = ConcatDataset(train_subsets)

val_dataset = datasets.ImageFolder(val_path, transform=val_transform)
val_subset = get_class_subset(val_dataset, class_indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)

def get_labels_from_loader(loader):
    labels = []
    for _, batch_labels in loader:
        labels.extend(batch_labels.numpy())
    return np.array(labels)

y_train = get_labels_from_loader(train_loader)
y_val = get_labels_from_loader(val_loader)

unique_labels = np.unique(y_train)
label_mapping = {old_label: new_idx for new_idx, old_label in enumerate(unique_labels)}

y_train = np.array([label_mapping[lbl] for lbl in y_train])
y_val = np.array([label_mapping[lbl] for lbl in y_val])

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}


Selected classes: ['n01632777', 'n01984695', 'n01728572', 'n01514859', 'n01614925', 'n01582220', 'n01806143', 'n01819313']


In [14]:
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

Using cache found in C:\Users\HP/.cache\torch\hub\facebookresearch_dino_main


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
  (head): Identity()
)

In [15]:
@torch.no_grad()
def collect_model_outputs(model, dataloader, max_batches=None):
    outputs = []
    for i, (images, _) in enumerate(dataloader):
        if max_batches and i >= max_batches:
            break
        images = images.to(device)
        out = model(images) 
        outputs.append(out.detach().cpu())
    return torch.cat(outputs, dim=0)

In [16]:
print("Collecting train outputs...")
train_outputs = collect_model_outputs(model, train_loader)

print("Collecting val outputs...")
val_outputs = collect_model_outputs(model, val_loader)

In [17]:
X_train = train_outputs.numpy()
X_val = val_outputs.numpy()

print("Training classifier on model output...")

clf = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    multi_class='multinomial',
    class_weight=class_weight_dict
)
clf.fit(X_train, y_train)

y_val_pred = clf.predict(X_val)

acc = accuracy_score(y_val, y_val_pred)
prec = precision_score(y_val, y_val_pred, average='macro', zero_division=0)
rec = recall_score(y_val, y_val_pred, average='macro', zero_division=0)
f1 = f1_score(y_val, y_val_pred, average='macro', zero_division=0)

print(f"\n→ Val Accuracy: {acc:.4f}")
print(f"→ Val Precision: {prec:.4f}")
print(f"→ Val Recall: {rec:.4f}")
print(f"→ Val F1-score: {f1:.4f}")

os.makedirs("linear_probes", exist_ok=True)
joblib.dump(clf, "linear_probes/final_linear_probe.joblib")

metrics_dict = {
    'accuracy': acc,
    'precision': prec,
    'recall': rec,
    'f1_score': f1
}

with open("metrics_dict.json", "w") as f:
    json.dump(metrics_dict, f, indent=4)

Training classifier on model output...


c:\Users\HP\studia\trzeci rok\semestr6\Warsztaty Badawcze\proekt\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



→ Val Accuracy: 0.2000
→ Val Precision: 0.1881
→ Val Recall: 0.2000
→ Val F1-score: 0.1877
